In [1]:
import pandas as pd
import yfinance as yf
import datetime

def ichimoku_cloud(data):
    """Calculate Ichimoku Cloud components."""
    data = data.copy()
    required_columns = ['High', 'Low', 'Close']
    if not all(col in data.columns for col in required_columns):
        raise KeyError(f"DataFrame missing required columns: {required_columns}")
    
    # Tenkan-Sen (Conversion Line): (9-period high + 9-period low) / 2
    data['Tenkan_Sen'] = (data['High'].rolling(window=9).max() + data['Low'].rolling(window=9).min()) / 2
    # Kijun-Sen (Baseline): (26-period high + 26-period low) / 2
    data['Kijun_Sen'] = (data['High'].rolling(window=26).max() + data['Low'].rolling(window=26).min()) / 2
    # Senkou Span A (Leading Span A): (Tenkan-Sen + Kijun-Sen) / 2, shifted forward 26 periods
    data['Senkou_Span_A'] = ((data['Tenkan_Sen'] + data['Kijun_Sen']) / 2).shift(26)
    # Senkou Span B (Leading Span B): (52-period high + 52-period low) / 2, shifted forward 26 periods
    data['Senkou_Span_B'] = ((data['High'].rolling(window=52).max() + data['Low'].rolling(window=52).min()) / 2).shift(26)
    # Chikou Span (Lagging Span): Close shifted back 26 periods (not needed for criteria but included for completeness)
    data['Chikou_Span'] = data['Close'].shift(-26)
    
    return data

def analyze_stock(data, ticker):
    """Check if the stock meets the specified Ichimoku criteria as of the latest data point."""
    # Ensure we have enough data to calculate Ichimoku components
    if len(data) < 52:  # Need at least 52 days for Senkou Span B
        return None
    
    # Get the latest data point (most recent closing price)
    latest = data.iloc[-1]
    
    # Extract relevant values
    close = latest['Close'].item()
    tenkan_sen = latest['Tenkan_Sen'].item()
    kijun_sen = latest['Kijun_Sen'].item()
    senkou_span_a = latest['Senkou_Span_A'] if pd.notna(latest['Senkou_Span_A']) else float('inf')
    senkou_span_b = latest['Senkou_Span_B'] if pd.notna(latest['Senkou_Span_B']) else float('inf')
    
    # Criteria 1: Last closing price above Ichimoku Cloud, Baseline, and Conversion Line
    price_above_cloud = pd.notna(senkou_span_a) and pd.notna(senkou_span_b) and close > senkou_span_a and close > senkou_span_b
    price_above_baseline = close > kijun_sen
    price_above_conversion = close > tenkan_sen
    
    # Criteria 2: Conversion Line above Baseline (Tenkan-Sen > Kijun-Sen)
    conversion_above_baseline = tenkan_sen > kijun_sen
    
    # Check if all criteria are met
    meets_criteria = (price_above_cloud and price_above_baseline and price_above_conversion and conversion_above_baseline)
    
    if meets_criteria:
        return {
            "Ticker": ticker,
            "Last_Close": close,
            "Tenkan_Sen": tenkan_sen,
            "Kijun_Sen": kijun_sen,
            "Senkou_Span_A": senkou_span_a,
            "Senkou_Span_B": senkou_span_b,
            "Meets_Criteria": True
        }
    else:
        return None

# -------- MAIN EXECUTION --------
# Define the list of tickers (shortened for demonstration; use your full list)
tickers = [
    "A", "AAL", "AAPL", "ABBV", "ABNB", "ABT", "ACN", "ADBE", "ADI", "ADM", "ADP", "ADSK", 
    "AEE", "AEP", "AES", "AFL", "AGNC", "AIG", "AIZ", "AJG", "AKAM", "ALB", "ALLE", "ALGN", 
    "ALL", "AMAT", "AMD", "AME", "AMGN", "AMP", "AMT", "AMZN", "ANET", "ANSS", "AON", "APA", 
    "APD", "APH", "APTV", "ARE", "ARM", "ASML", "ATO", "AVB", "AVGO", "AVY", "AXON", "AXP", 
    "AZN", "AZO", "BALL", "BAC", "BA", "BAX", "BBY", "BDX", "BEN", "BF-B", "BIIB", "BIO", 
    "BK", "BKNG", "BKR", "BLK", "BMY", "BWA", "BXP", "C", "CAG", "CAH", "CARR", "CAT", 
    "CB", "CBOE", "CBRE", "CDAY", "CDNS", "CDW", "CE", "CEG", "CF", "CHD", "CHRW", "CHTR", 
    "CI", "CINF", "CL", "CLX", "CMA", "CMCSA", "CME", "CMG", "CMI", "CNP", "COF", "COST", 
    "CPB", "CPRT", "CRL", "CRWD", "CSCO", "CSGP", "CSX", "CTAS", "LII", "CTRA", "CTSH", 
    "CTVA", "CVS", "CVX", "D", "DAL", "DASH", "DD", "DDOG", "DE", "DFS", "DG", "DHI", "DHR", 
    "DIS", "DLTR", "DOV", "DOW", "DPZ", "APP", "DUK", "DVN", "DXCM", "EBAY", "ECL", "ED", 
    "EEFT", "EIX", "EL", "ELV", "EMN", "EMR", "ENPH", "EOG", "EPAM", "EQIX", "EQR", "ES", 
    "ESS", "ETN", "ETR", "ETSY", "EVRG", "EW", "EXC", "EXPD", "EXPE", "EXR", "F", "FANG", 
    "FAST", "FCX", "FE", "FFIV", "FIS", "FISV", "FITB", "FLT", "FMC", "FRT", "FTNT", 
    "FTV", "GD", "GE", "GEHC", "GILD", "GIS", "GL", "GLW", "GM", "GOOG", "GOOGL", "GPC", 
    "GPN", "GS", "HAL", "HAS", "HBAN", "HCA", "HD", "HES", "HIG", "HII", "HLT", "HOLX", 
    "HON", "HRL", "HSIC", "HSY", "HST", "HUM", "IBM", "ICE", "IDXX", "IEX", "ILMN", "INCY", 
    "INTC", "INTU", "IP", "IPG", "IQV", "IR", "IRM", "ISRG", "IT", "ITW", "IVZ", "J", 
    "JBHT", "JCI", "JEF", "JKHY", "JNJ", "JNPR", "JPM", "K", "KDP", "KEY", "KEYS", "KHC", 
    "KLAC", "KMB", "KMI", "KMX", "KO", "KR", "L", "LHX", "LH", "LIN", "LKQ", "LLY", "LMT", 
    "LNC", "LRCX", "LULU", "LUMN", "LUV", "LOW", "LYB", "MA", "MAR", "MAS", "MCD", "MCHP", 
    "MCK", "MDLZ", "MDT", "MEI", "MELI", "META", "MET", "MGM", "MKC", "MKTX", "MLM", "MMM", 
    "MNST", "MO", "MOS", "MPC", "MPWR", "MRK", "MRNA", "TPL", "MS", "MSCI", "MSFT", "MSI", 
    "MTB", "MTCH", "MTD", "MU", "NDAQ", "NDSN", "NEE", "NFLX", "NKE", "NOC", "NOW", "NRG", 
    "NSC", "NTAP", "NTRS", "NUE", "NVDA", "NVR", "NXPI", "O", "ODFL", "OKE", "OMC", "ON", 
    "ORLY", "OTIS", "OXY", "PANW", "PARA", "PAYC", "PAYX", "PCAR", "PDD", "PEG", 
    "PEP", "PFE", "PFG", "PG", "PGR", "PH", "PHM", "PLD", "PLTR", "PM", "PNC", "PNR", 
    "PNW", "POOL", "PPG", "PPL", "PRU", "PSA", "PTC", "PWR", "vst", "PYPL", "QCOM", "QRVO", 
    "RCL", "REG", "REGN", "RF", "RHI", "RJF", "RL", "RMD", "ROK", "ROL", "ROP", "ROST", 
    "RSG", "RTX", "SBAC", "SBUX", "SCHW", "SHEL", "SHW", "SJM", "SLB", "SNA", "SNPS", 
    "SO", "SPG", "SRE", "STE", "STT", "STX", "STZ", "SWK", "SWKS", "SYF", "SYK", "SYY", 
    "T", "TAP", "TDG", "TDY", "TECH", "TEL", "TER", "TGT", "TJX", "TMO", "TMUS", "TPR", 
    "TRGP", "TSCO", "TSLA", "TSN", "TT", "TTD", "TTWO", "TXN", "TXT", "TYL", "UAL", "UDR", 
    "UHS", "ULTA", "UNH", "UNP", "UPS", "URI", "USB", "V", "VFC", "VICI", "VLO", "VMC", 
    "VRSK", "VRTX", "VTR", "VTRS", "VZ", "WAB", "WAT", "WBA", "WBD", "WDC", "WEC", "WELL", 
    "WFC", "WHR", "WM", "WMB", "WMT", "WRB", "WST", "WTW", "WY", "WYNN", "XEL", 
    "XOM", "XRAY", "XYL", "YUM", "ZBH", "ZBRA", "ZION", "ZTS", "ZS", "MSTR"
]

# Define the date range for data (need enough historical data for Ichimoku calculations)
end_date = datetime.datetime(2025, 3, 13)  # Today
start_date = end_date - datetime.timedelta(days=180)  # 6 months of data to ensure enough for Ichimoku calculations

results = []

for ticker in tickers:
    print(f"\nDownloading data for {ticker}...")
    try:
        # Download data up to today (March 13, 2025)
        df = yf.download(ticker, start=start_date, end=end_date, interval="1d", auto_adjust=True)
    except Exception as e:
        print(f"Failed to download data for {ticker}: {e}")
        continue

    if df.empty:
        print(f"No data available for {ticker}.")
        continue

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    # Calculate Ichimoku Cloud components
    df = ichimoku_cloud(df)
    df = df.dropna(subset=['Close', 'High', 'Low'])  # Drop rows missing core data

    # Analyze the stock based on the criteria
    result = analyze_stock(df, ticker)
    if result:
        results.append(result)

# Output the filtered stocks
if results:
    results_df = pd.DataFrame(results)
    print("\nStocks meeting the Ichimoku criteria as of March 13, 2025:")
    print(results_df[["Ticker", "Last_Close", "Tenkan_Sen", "Kijun_Sen", "Senkou_Span_A", "Senkou_Span_B"]])
    
    # Export to CSV
    results_df.to_csv("stocks_meeting_ichimoku_criteria_2025-03-13.csv", index=False)
    print(f"\nExported {len(results_df)} stocks to 'stocks_meeting_ichimoku_criteria_2025-03-13.csv'.")
else:
    print("\nNo stocks met the specified Ichimoku criteria.")

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


...

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed




Stocks meeting the Ichimoku criteria as of March 13, 2025:
  Ticker  Last_Close  Tenkan_Sen   Kijun_Sen  Senkou_Span_A  Senkou_Span_B
0   ABBV  212.059998  211.535004  202.855003     179.915001     177.024840
1    ALL  198.250000  198.215004  193.709833     186.779838     191.746272
2    CME  256.720001  255.966139  250.299164     233.480479     227.841006
3    CNP   34.549999   34.125000   33.381871      31.917681      31.242389
4    ELV  411.040009  407.781342  401.975687     392.202694     387.226055
5     MO   57.810001   57.469999   55.839998      51.945002      53.506412
6    NEE   72.919998   72.654999   71.634060      69.921370      72.041397
7   SHEL   67.379997   66.799999   66.508636      64.206540      63.035372
8   VRTX  495.420013  488.950012  474.945007     441.982506     428.895004
9    WBA   11.200000   10.835000   10.585000      10.992500      10.665000

Exported 10 stocks to 'stocks_meeting_ichimoku_criteria_2025-03-13.csv'.
